In [70]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import numpy as np
import pandas as pd
from collections import defaultdict
import random
from google.colab import drive

In [71]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [72]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [73]:
CHECKPOINT_BASE = "/content/drive/MyDrive/DeBERTa_ANLI_Checkpoint"
MODEL_PATH = f"{CHECKPOINT_BASE}/best_model"  # Use best_model instead
MAX_LENGTH = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LABEL_MAP = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
SAMPLES_PER_LABEL = 4  # 4 samples per label = 12 total per split

print("="*80)
print("MODEL QUALITY EVIDENCE ANALYSIS".center(80))
print("="*80)

                        MODEL QUALITY EVIDENCE ANALYSIS                         


In [74]:
print("\nLoading model and tokenizer...")

# Define checkpoint paths in priority order
import os
import glob

checkpoint_paths = [
    f"{CHECKPOINT_BASE}/final_model",
    f"{CHECKPOINT_BASE}/best_model"
]

# Also check for epoch checkpoints
epoch_checkpoints = sorted(glob.glob(f"{CHECKPOINT_BASE}/checkpoint_epoch_*"))
if epoch_checkpoints:
    checkpoint_paths.extend(epoch_checkpoints[::-1])

# Try to load from each path
MODEL_PATH = None
for path in checkpoint_paths:
    if os.path.exists(path):
        MODEL_PATH = path
        break

if MODEL_PATH is None:
    raise FileNotFoundError(f"No model checkpoint found in {CHECKPOINT_BASE}")

# Try loading tokenizer with multiple strategies
tokenizer = None
strategies = [
    ("Direct from checkpoint", lambda: AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=False)),
    ("Direct from checkpoint (trust remote)", lambda: AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)),
    ("From base model", lambda: AutoTokenizer.from_pretrained('microsoft/deberta-v3-large')),
]

for strategy_name, loader_func in strategies:
    try:
        tokenizer = loader_func()
        break
    except Exception as e:
        continue

if tokenizer is None:
    raise RuntimeError("Failed to load tokenizer with all strategies")

# Try loading model with multiple strategies
model = None
model_strategies = [
    ("Direct from checkpoint", lambda: AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, trust_remote_code=False, local_files_only=True)),
    ("Without local_files_only", lambda: AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, trust_remote_code=False)),
    ("With trust_remote_code", lambda: AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, trust_remote_code=True)),
]

# If final_model fails, try best_model
if MODEL_PATH == f"{CHECKPOINT_BASE}/final_model":
    best_model_path = f"{CHECKPOINT_BASE}/best_model"
    if os.path.exists(best_model_path):
        model_strategies.append((
            f"Fallback to best_model",
            lambda: AutoModelForSequenceClassification.from_pretrained(best_model_path)
        ))

for strategy_name, loader_func in model_strategies:
    try:
        model = loader_func()

        # Update MODEL_PATH if we used a fallback
        if "Fallback to best_model" in strategy_name:
            MODEL_PATH = f"{CHECKPOINT_BASE}/best_model"

        break
    except Exception as e:
        continue

if model is None:
    raise RuntimeError("Failed to load model with all strategies")

model.to(DEVICE)
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

print("Loading ANLI R2 dataset...")
dataset = load_dataset('anli')
splits = {
    'train': dataset['train_r2'],
    'dev': dataset['dev_r2'],
    'test': dataset['test_r2']
}
print("Dataset loaded")


Loading model and tokenizer...
Model loaded: 435.06M parameters
Loading ANLI R2 dataset...
Dataset loaded


In [75]:
def stratified_sample(dataset, n_per_label=4, seed=42):
    """
    Perform stratified sampling to get n samples per label class.

    Args:
        dataset: HuggingFace dataset split
        n_per_label: Number of samples per label (default: 2)
        seed: Random seed for reproducibility

    Returns:
        List of sampled indices
    """
    # Group indices by label
    label_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        label = dataset[idx]['label']
        label_to_indices[label].append(idx)

    # Sample n_per_label from each label class
    sampled_indices = []
    random.seed(seed)

    for label in sorted(label_to_indices.keys()):
        indices = label_to_indices[label]
        # Sample without replacement
        selected = random.sample(indices, min(n_per_label, len(indices)))
        sampled_indices.extend(selected)

    return sorted(sampled_indices)

In [76]:
def predict_sample(premise, hypothesis, model, tokenizer, device, max_length=256):
    """
    Make prediction for a single premise-hypothesis pair.

    Returns:
        dict with prediction, probabilities, and confidence
    """
    model.eval()

    # Tokenize
    encoding = tokenizer(
        premise,
        hypothesis,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Predict
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits[0]
        probs = torch.softmax(logits, dim=-1)
        pred_id = torch.argmax(logits).item()
        confidence = probs[pred_id].item()

    return {
        'predicted_label_id': pred_id,
        'predicted_label': LABEL_MAP[pred_id],
        'confidence': confidence,
        'probabilities': {
            'entailment': probs[0].item(),
            'neutral': probs[1].item(),
            'contradiction': probs[2].item()
        }
    }

In [77]:
print("\n" + "="*80)
print("PERFORMING STRATIFIED SAMPLING AND PREDICTIONS".center(80))
print("="*80)

all_results = []

for split_name, split_data in splits.items():
    print(f"\nProcessing {split_name.upper()} split...")

    # Perform stratified sampling
    sampled_indices = stratified_sample(split_data, n_per_label=SAMPLES_PER_LABEL, seed=SEED)

    # Make predictions
    for idx in sampled_indices:
        sample = split_data[idx]
        premise = sample['premise']
        hypothesis = sample['hypothesis']
        true_label_id = sample['label']
        true_label = LABEL_MAP[true_label_id]

        # Get prediction
        prediction = predict_sample(premise, hypothesis, model, tokenizer, DEVICE, MAX_LENGTH)

        # Determine correctness
        is_correct = (prediction['predicted_label_id'] == true_label_id)

        # Store result
        result = {
            'split': split_name.upper(),
            'sample_index': idx,
            'premise': premise,
            'hypothesis': hypothesis,
            'true_label': true_label,
            'predicted_label': prediction['predicted_label'],
            'is_correct': is_correct,
            'confidence': prediction['confidence'],
            'prob_entailment': prediction['probabilities']['entailment'],
            'prob_neutral': prediction['probabilities']['neutral'],
            'prob_contradiction': prediction['probabilities']['contradiction']
        }
        all_results.append(result)

print(f"\nCompleted all predictions")


                 PERFORMING STRATIFIED SAMPLING AND PREDICTIONS                 

Processing TRAIN split...

Processing DEV split...

Processing TEST split...

Completed all predictions


In [78]:
results_df = pd.DataFrame(all_results)

# Calculate accuracy per split
print("\n" + "="*80)
print("ACCURACY BY SPLIT".center(80))
print("="*80)
for split_name in ['TRAIN', 'DEV', 'TEST']:
    split_df = results_df[results_df['split'] == split_name]
    accuracy = split_df['is_correct'].mean()
    correct = split_df['is_correct'].sum()
    total = len(split_df)
    print(f"{split_name:10s}: {correct}/{total} correct = {accuracy:.2%}")

overall_accuracy = results_df['is_correct'].mean()
overall_correct = results_df['is_correct'].sum()
overall_total = len(results_df)
print(f"{'─'*80}")
print(f"{'OVERALL':10s}: {overall_correct}/{overall_total} correct = {overall_accuracy:.2%}")
print("="*80 + "\n")


                               ACCURACY BY SPLIT                                
TRAIN     : 12/12 correct = 100.00%
DEV       : 7/12 correct = 58.33%
TEST      : 9/12 correct = 75.00%
────────────────────────────────────────────────────────────────────────────────
OVERALL   : 28/36 correct = 77.78%



In [79]:
print("\n" + "="*80)
print("DETAILED SAMPLE ANALYSIS".center(80))
print("="*80)

for idx, row in results_df.iterrows():
    sample_num = idx + 1
    status = "CORRECT" if row['is_correct'] else "INCORRECT"

    print(f"\n{'-'*80}")
    print(f"SAMPLE #{sample_num}/18 [{row['split']}] - {status}")
    print(f"{'-'*80}")

    print(f"\nPREMISE:")
    print(f"   {row['premise']}")

    print(f"\nHYPOTHESIS:")
    print(f"   {row['hypothesis']}")

    print(f"\nLABELS:")
    print(f"   True Label:      {row['true_label'].upper()}")
    print(f"   Predicted Label: {row['predicted_label'].upper()}")
    print(f"   Status:          {status}")

    print(f"\nMODEL CONFIDENCE:")
    print(f"   Entailment:    {row['prob_entailment']:.4f} ({row['prob_entailment']*100:.2f}%)")
    print(f"   Neutral:       {row['prob_neutral']:.4f} ({row['prob_neutral']*100:.2f}%)")
    print(f"   Contradiction: {row['prob_contradiction']:.4f} ({row['prob_contradiction']*100:.2f}%)")
    print(f"   Prediction Confidence: {row['confidence']:.4f} ({row['confidence']*100:.2f}%)")

    print(f"\nANALYSIS:")
    if row['is_correct']:
        conf_level = 'HIGH' if row['confidence'] > 0.7 else 'MODERATE' if row['confidence'] > 0.5 else 'LOW'
        print(f"   The model correctly identified this as {row['true_label'].upper()}.")
        print(f"   Confidence level: {conf_level}")
    else:
        print(f"   The model incorrectly predicted {row['predicted_label'].upper()} instead of {row['true_label'].upper()}.")
        print(f"   This may indicate difficulty with subtle semantic distinctions.")
        print(f"   Confidence in wrong prediction: {row['confidence']:.2%}")

print(f"\n{'='*80}")
print("END OF DETAILED ANALYSIS".center(80))
print("="*80)


                            DETAILED SAMPLE ANALYSIS                            

--------------------------------------------------------------------------------
SAMPLE #1/18 [TRAIN] - CORRECT
--------------------------------------------------------------------------------

PREMISE:
   The End Tour was the farewell tour of the heavy metal band Black Sabbath, featuring founding members Ozzy Osbourne, Tony Iommi and Geezer Butler. The tour concluded Sabbath's 40+ year career. The final show was February 4, 2017, in their home city of Birmingham, UK.

HYPOTHESIS:
   Some bands that come from places like Birmingham, UK have had careers that have lasted longer than others, some even extending beyond 2, 3, or 4 sets of ten years.

LABELS:
   True Label:      ENTAILMENT
   Predicted Label: ENTAILMENT
   Status:          CORRECT

MODEL CONFIDENCE:
   Entailment:    0.9994 (99.94%)
   Neutral:       0.0004 (0.04%)
   Contradiction: 0.0002 (0.02%)
   Prediction Confidence: 0.9994 (99.94%)

ANA

In [80]:
output_path = "/content/drive/MyDrive/DeBERTa_ANLI_Checkpoint/stratified_sample_results.csv"
results_df.to_csv(output_path, index=False)
print(f"\nResults saved to: {output_path}")

# Also save a formatted text report
report_path = "/content/drive/MyDrive/DeBERTa_ANLI_Checkpoint/stratified_sample_report.txt"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("="*80 + "\n")
    f.write("MODEL QUALITY EVIDENCE: STRATIFIED SAMPLE ANALYSIS\n")
    f.write("="*80 + "\n\n")

    f.write(f"Model: DeBERTa-v3-Large\n")
    f.write(f"Dataset: ANLI R2\n")
    f.write(f"Sampling Strategy: Stratified (2 samples per label per split)\n")
    f.write(f"Total Samples: 18 (6 per split x 3 splits)\n")
    f.write(f"Random Seed: {SEED}\n\n")

    f.write("="*80 + "\n")
    f.write("OVERALL PERFORMANCE\n")
    f.write("="*80 + "\n\n")

    for split_name in ['TRAIN', 'DEV', 'TEST']:
        split_df = results_df[results_df['split'] == split_name]
        accuracy = split_df['is_correct'].mean()
        f.write(f"{split_name:10s}: {split_df['is_correct'].sum()}/{len(split_df)} = {accuracy:.2%}\n")

    f.write(f"{'-'*80}\n")
    f.write(f"{'OVERALL':10s}: {overall_correct}/{overall_total} = {overall_accuracy:.2%}\n\n")

    for idx, row in results_df.iterrows():
        sample_num = idx + 1
        status = "CORRECT" if row['is_correct'] else "INCORRECT"

        f.write("\n" + "-"*80 + "\n")
        f.write(f"SAMPLE #{sample_num}/18 [{row['split']}] - {status}\n")
        f.write("-"*80 + "\n\n")

        f.write(f"PREMISE:\n{row['premise']}\n\n")
        f.write(f"HYPOTHESIS:\n{row['hypothesis']}\n\n")

        f.write(f"LABELS:\n")
        f.write(f"  True:      {row['true_label'].upper()}\n")
        f.write(f"  Predicted: {row['predicted_label'].upper()}\n")
        f.write(f"  Status:    {status}\n\n")

        f.write(f"CONFIDENCE:\n")
        f.write(f"  Entailment:    {row['prob_entailment']:.4f}\n")
        f.write(f"  Neutral:       {row['prob_neutral']:.4f}\n")
        f.write(f"  Contradiction: {row['prob_contradiction']:.4f}\n\n")

print(f"Text report saved to: {report_path}")


Results saved to: /content/drive/MyDrive/DeBERTa_ANLI_Checkpoint/stratified_sample_results.csv
Text report saved to: /content/drive/MyDrive/DeBERTa_ANLI_Checkpoint/stratified_sample_report.txt


In [81]:
print("\n" + "="*80)
print("SUMMARY STATISTICS".center(80))
print("="*80)

print("\nDISTRIBUTION OF PREDICTIONS:")
print(f"{'-'*80}")
print(f"{'Label':<20s} {'Count':<10s} {'Percentage':<15s}")
print(f"{'-'*80}")
for label in ['entailment', 'neutral', 'contradiction']:
    count = (results_df['predicted_label'] == label).sum()
    pct = count / len(results_df) * 100
    print(f"{label.capitalize():<20s} {count:<10d} {pct:.1f}%")
print(f"{'-'*80}")

print("\nCONFIDENCE DISTRIBUTION:")
print(f"{'-'*80}")
print(f"Mean Confidence:   {results_df['confidence'].mean():.4f}")
print(f"Median Confidence: {results_df['confidence'].median():.4f}")
print(f"Min Confidence:    {results_df['confidence'].min():.4f}")
print(f"Max Confidence:    {results_df['confidence'].max():.4f}")
print(f"{'-'*80}")

print("\nCONFIDENCE BY CORRECTNESS:")
print(f"{'-'*80}")
correct_conf = results_df[results_df['is_correct']]['confidence'].mean()
incorrect_conf = results_df[~results_df['is_correct']]['confidence'].mean()
print(f"Correct predictions:   {correct_conf:.4f}")
print(f"Incorrect predictions: {incorrect_conf:.4f}")
print(f"{'-'*80}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print("\nFiles saved:")
print(f"  1. CSV: {output_path}")
print(f"  2. Report: {report_path}")
print("="*80)


                               SUMMARY STATISTICS                               

DISTRIBUTION OF PREDICTIONS:
--------------------------------------------------------------------------------
Label                Count      Percentage     
--------------------------------------------------------------------------------
Entailment           14         38.9%
Neutral              10         27.8%
Contradiction        12         33.3%
--------------------------------------------------------------------------------

CONFIDENCE DISTRIBUTION:
--------------------------------------------------------------------------------
Mean Confidence:   0.9785
Median Confidence: 0.9989
Min Confidence:    0.7802
Max Confidence:    0.9999
--------------------------------------------------------------------------------

CONFIDENCE BY CORRECTNESS:
--------------------------------------------------------------------------------
Correct predictions:   0.9783
Incorrect predictions: 0.9793
----------------------